# Retail Business Analytics — Task 5

This notebook reproduces the core analysis for the UCI Online Retail dataset.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

DATA = Path('Online Retail.xlsx')
df = pd.read_excel(DATA)
df.head()

## Data quality checks

In [ ]:
df.info()
print('Exact duplicates:', df.duplicated().sum())
print('Missing values:\n', df.isna().sum())

## Prepare positive-sales analysis table

In [ ]:
df['InvoiceNo'] = df['InvoiceNo'].astype(str)
df['IsCancellation'] = df['InvoiceNo'].str.startswith('C')
sales = df.drop_duplicates().loc[lambda x: (~x['IsCancellation']) & (x['Quantity'] > 0) & (x['UnitPrice'] > 0)].copy()
sales['Revenue'] = sales['Quantity'] * sales['UnitPrice']
sales['Month'] = sales['InvoiceDate'].dt.to_period('M').astype(str)
sales.head()

## Business KPIs

In [ ]:
revenue = sales['Revenue'].sum()
orders = sales['InvoiceNo'].nunique()
print(f'Revenue: £{revenue:,.2f}')
print('Orders:', orders)
print(f'Average order value: £{revenue/orders:,.2f}')
print('Customers with IDs:', sales['CustomerID'].nunique())

## Monthly sales trend

In [ ]:
monthly = sales.groupby('Month')['Revenue'].sum().sort_index()
monthly.plot(kind='bar', figsize=(10, 5), title='Monthly Sales Revenue', ylabel='Revenue (GBP)', xlabel='Month', rot=45)
plt.tight_layout()

## Country and product rankings

In [ ]:
display(sales.groupby('Country')['Revenue'].sum().sort_values(ascending=False).head(10))
display(sales.dropna(subset=['Description']).groupby('Description')['Revenue'].sum().sort_values(ascending=False).head(10))